# TRIBE v2 Demo: Predicting Brain Responses to Naturalistic Stimuli

[TRIBE v2](https://github.com/facebookresearch/tribev2) is a deep multimodal brain encoding model that predicts **fMRI brain responses** to naturalistic stimuli — video, audio, and text.

It combines state-of-the-art feature extractors — **LLaMA 3.2** (text), **V-JEPA2** (video), and **Wav2Vec-BERT** (audio) — into a unified Transformer that maps multimodal representations onto the cortical surface (**fsaverage5**, ~20k vertices).

This notebook demos the **text** path only — the modality NeuroTutorSim actually uses. The video path is intentionally omitted: decoding a video clip saturates Colab free's ~12.7 GB of system RAM and kills the runtime, and V-JEPA video features are irrelevant to a text-based educational corpus.

In this notebook, we will:
1. Load a pretrained TRIBE v2 model from HuggingFace
2. Predict brain responses to **audio** generated from text (via text-to-speech)
3. Visualize the predicted activity on a 3D brain surface

## Setup (for Colab users)

1. Activate the GPU: **Runtime → Change runtime type → GPU**.
2. Run the setup cell below. It installs the **exact** Track A dependency set frozen in the repo's `uv.lock` — torch, a **matched** torchaudio, transformers, and TRIBE v2 pinned to a specific commit. Installing the locked versions is what keeps this reproducible (brief §4.3) and guarantees the torch/torchaudio ABI agrees (a mismatch is what causes `undefined symbol: aoti_torch_abi_version`).
3. When it finishes, **Runtime → Restart session** so the newly installed packages are picked up, then run the cells **below** the setup cell (skip the setup cell on the second pass — the packages are already installed).

In [ ]:
# ── Setup (Colab): install the exact Track A stack pinned in the repo's uv.lock ──
# When this finishes:  Runtime → Restart session,  then run the cells BELOW this one.
import os
import subprocess
import sys

REPO = "https://github.com/MatteoGuardamagna4/neurotutorsim.git"
REPO_DIR = "/content/neurotutorsim"

# torch/torchaudio/torchvision are pinned to CUDA 12.4 builds (e.g. 2.6.0+cu124).
# Those "+cu124" wheels live ONLY on the PyTorch index, not PyPI, and `uv export`
# does not bake that index into the requirements file — so the install step must
# add it explicitly. `unsafe-first-match` keeps every other package on PyPI and
# only falls through to the PyTorch index for the exact versions PyPI lacks.
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu124"


def sh(cmd, **kw):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, **kw)


sh([sys.executable, "-m", "pip", "install", "-q", "uv"])

if not os.path.isdir(REPO_DIR):
    sh(["git", "clone", "--depth", "1", REPO, REPO_DIR])

sh([
    "uv", "export", "--frozen", "--extra", "tribe",
    "--no-hashes", "--no-emit-project", "-o", "/tmp/tribe-locked.txt",
], cwd=REPO_DIR)

sh([
    "uv", "pip", "install", "--system",
    "--index-strategy", "unsafe-first-match",
    "--extra-index-url", PYTORCH_INDEX,
    "-r", "/tmp/tribe-locked.txt",
])

print("\n  Locked Track A stack installed.")
print("   Now: Runtime → Restart session, then run the cells BELOW (skip this one).")

## Loading the model

We load TRIBE v2 model from [HuggingFace Hub](https://huggingface.co/facebook/tribev2). On the first run, this downloads the model checkpoint and config (~1 GB). Subsequent runs use the cached version.

We also initialize a `PlotBrain` object for 3D brain surface visualization using the **fsaverage5** mesh.

In [23]:
from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain
from pathlib import Path

CACHE_FOLDER = Path("./cache")

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)
plotter = PlotBrain(mesh="fsaverage5")

/private/home/sdascoli/tribev2/.venv/lib/python3.12/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-03-30 15:11:44 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /private/home/sdascoli/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt


## Persist outputs to Google Drive

Colab has no persistent disk — the runtime (and `./cache`) is wiped on disconnect. We mount Google Drive and write outputs to **`My Drive/NeuroTutorSim/tribe_demo`**:

- **prediction arrays** → `*_preds.parquet` (rows = timesteps, columns = vertices)
- **brain figures** → `*.png`

Writes are **idempotent**: a file that already exists is skipped rather than silently regenerated (pass `overwrite=True` to force). Off Colab, outputs fall back to a local `./drive_local` folder so the notebook still runs.

In [ ]:
# --- Persist outputs to Google Drive ---
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
except ModuleNotFoundError:
    # Not on Colab: fall back to a local folder so the notebook still runs.
    DRIVE_ROOT = Path("./drive_local")

OUT_DIR = DRIVE_ROOT / "NeuroTutorSim" / "tribe_demo"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be saved to: {OUT_DIR}")


def save_preds_parquet(preds, name, overwrite=False):
    """Save a (n_timesteps, n_vertices) prediction array as parquet.

    Rows = timesteps, columns = vertex_00000 ... vertex_NNNNN.
    Reload with: pd.read_parquet(path).values  -> recovers the array.
    """
    path = OUT_DIR / f"{name}_preds.parquet"
    if path.exists() and not overwrite:
        print(f"[skip] {path.name} already exists (pass overwrite=True to regenerate)")
        return path
    arr = np.asarray(preds)
    cols = [f"vertex_{v:05d}" for v in range(arr.shape[1])]
    df_out = pd.DataFrame(arr, columns=cols)
    df_out.index.name = "timestep"
    df_out.to_parquet(path)
    print(f"[saved] {path}  shape={arr.shape}")
    return path


def save_figure(fig, name, overwrite=False, dpi=150):
    """Save a matplotlib figure as PNG."""
    path = OUT_DIR / f"{name}.png"
    if path.exists() and not overwrite:
        print(f"[skip] {path.name} already exists (pass overwrite=True to regenerate)")
        return path
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"[saved] {path}")
    return path

## Fix: whisperx transcription on CPU

To recover precise word-level timings, TRIBE runs [whisperx](https://github.com/m-bain/whisperx) in a throwaway environment via `uvx`. That environment lands on the **CPU**, where the ctranslate2 Whisper backend supports only `int8`/`float32` — not the `float16` TRIBE requests by default, which raises `ValueError: … do not support efficient float16 computation`.

The cell below transparently rewrites any whisperx invocation to `--compute_type int8` (valid on CPU **and** GPU, accurate enough for word alignment). Run it once per session, after the model is loaded and before the prediction cells.

In [ ]:
# ── Fix: whisperx float16-on-CPU crash ──
# To recover word-level timings, TRIBE shells out to `uvx whisperx … --compute_type float16`.
# That throwaway whisperx env runs the ctranslate2 Whisper model on CPU, and ctranslate2
# supports float16 only on GPU → ValueError. int8 is CPU- *and* GPU-safe and accurate enough
# for alignment, so we transparently rewrite any whisperx call to --compute_type int8.
# Run once per session, after the model is loaded and before the prediction cells.
import subprocess as _sp
import tribev2.eventstransforms as _ets


class _WhisperxCPUSafe:
    """Proxy for the `subprocess` module used inside tribev2.eventstransforms:
    forces whisperx to --compute_type int8; every other call passes straight through."""

    def __getattr__(self, name):
        return getattr(_sp, name)

    def run(self, cmd, *args, **kwargs):
        if isinstance(cmd, (list, tuple)) and "whisperx" in cmd:
            cmd = list(cmd)
            if "--compute_type" in cmd:
                cmd[cmd.index("--compute_type") + 1] = "int8"
            else:
                cmd += ["--compute_type", "int8"]
        return _sp.run(cmd, *args, **kwargs)


_ets.subprocess = _WhisperxCPUSafe()
print("Patched whisperx → compute_type=int8 (CPU-safe).")

In [ ]:
# ── Fix OOM: keep the Llama-3.2-3B text encoder entirely in VRAM ──
# neuralset's HuggingFaceText._load_model loads the text encoder on its
# device == "accelerate" branch with device_map="auto". accelerate then leaves a
# VRAM safety margin and offloads part of the 3B model to CPU → those ~6 GB land in
# the runtime's ~12.7 GB of system RAM and saturate it, so Colab kills the session.
# That load goes through `transformers.AutoModel.from_pretrained` (Llama-3.2-3B is
# not t5/bert/Phi-3/Llama-Vision, so neuralset uses plain AutoModel). We wrap that
# classmethod and rewrite device_map="auto" → {"": 0} so the whole encoder stays on
# GPU 0 (T4 has 15 GB — plenty) and system RAM stays free. Non-"auto" loads pass
# through untouched.
import torch
from transformers import AutoModel

if not getattr(AutoModel, "_forced_gpu_patch", False):
    _orig_from_pretrained = AutoModel.from_pretrained.__func__  # unwrap classmethod

    def _load_on_gpu(cls, *args, **kwargs):
        if kwargs.get("device_map") == "auto":
            torch.cuda.empty_cache()         # release VRAM cached by the previous extractor
            kwargs["device_map"] = {"": 0}    # everything on GPU 0 — no CPU/RAM offload
        return _orig_from_pretrained(cls, *args, **kwargs)

    AutoModel.from_pretrained = classmethod(_load_on_gpu)
    AutoModel._forced_gpu_patch = True
    print("Patched: AutoModel device_map='auto' → {'': 0} (text encoder stays in VRAM).")
else:
    print("Patch already applied.")

## Predict brain responses to text (via text-to-speech)

TRIBE v2 can also predict brain responses to **text** input. Since the model was trained on naturalistic audio/video stimuli, text is first converted to speech using Google Text-to-Speech (gTTS), then transcribed back to obtain precise word-level timings.

Below, we use a passage from Shakespeare's *Hamlet* as input.

In [27]:
text = """
To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die, to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
"""

text_path = CACHE_FOLDER / "shakespeare.txt"
text_path.write_text(text)

df = model.get_events_dataframe(text_path=text_path)
display(df.head(8)[["type", "start", "duration", "filepath", "text", "context"]])

,type,start,duration,filepath,text,context
0,Audio,0.000000,23.256000,cache/tribev2.demo_utils.TextToEvents.get_even...,NaN,
1,Sentence,0.090999,1.261002,NaN,To be or not to be.,
2,Text,0.091000,22.590000,NaN,To be or not to be. That is the question. Whet...,
3,Word,0.091000,0.100000,NaN,To,To
4,Word,0.271000,0.200000,NaN,be,To be
5,Word,0.551000,0.060000,NaN,or,To be or
6,Word,0.691000,0.200000,NaN,not,To be or not
7,Word,0.931000,0.100000,NaN,to,To be or not to


### Run the model

We pass the events dataframe to `model.predict()`, which extracts features for each modality, runs them through the Transformer, and returns predicted brain activity.

NOTE: you will have to request access to the Llama-3.2 model using your HuggingFace account.

The output `preds` has shape `(n_timesteps, n_vertices)` — one prediction per second of stimulus, with ~20k cortical vertices. The `segments` list contains the corresponding time segments with their associated events.

In [ ]:
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")

save_preds_parquet(preds, name="text_hamlet")

### Visualize predictions on the brain surface

We visualize the first 15 seconds of predicted activity on the fsaverage5 cortical mesh. For audio-only stimuli, the stimulus display shows the spoken words at each time step.

In [ ]:
n_timesteps = 15
fig = plotter.plot_timesteps(preds[:n_timesteps], segments=segments[:n_timesteps], cmap="fire", norm_percentile=99, vmin=.6, alpha_cmap=(0, .2), show_stimuli=True)

save_figure(fig, name="text_hamlet_timesteps")